# Single Agent - Langchain


In the coming examples, we will build an agent capable of explaining any topic via three mediums: text, image, or video. More specifically, based on the question asked, the agent will decide whether to explain the topic in what format.

## Defining tools


The first step after configuring our environment is defining the tools we will give to our agent. Let’s import them:


In [1]:
!pip install langchain langchain_openai langchain_groq langchain_community langgraph ipykernel python-dotenv
!pip install langgraph-checkpoint-sqlite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.7/123.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.9/565.9 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
ERROR: pip's dependency r

In [2]:
!pip install wikipedia youtube_search requests

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=2e1b93a311b3fe231a839020408c94b71674f14a755891118b0cc953e91c7315
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [5]:
import gdown
url = 'https://drive.google.com/file/d/1f-X_cbCcJG0JrJl2FsCNceuA6POKjnXm/view?usp=drive_link'
output_path = '.env'
gdown.download(url, output_path, quiet=False, fuzzy=True)

Downloading...
From: https://drive.google.com/uc?id=1f-X_cbCcJG0JrJl2FsCNceuA6POKjnXm
To: /content/.env
100%|██████████| 71.0/71.0 [00:00<00:00, 190kB/s]


'.env'

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()
# GROQ_API_KEY powers the Groq chat model (the agent's "brain")
groq_api_key = os.getenv('GROQ_API_KEY')

In [8]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessageChunk, SystemMessage
# Initialize the model (Groq)
# 'meta-llama/llama-4-scout-17b-16e-instruct' is used here because
# 'llama-3.3-70b-versatile' repeatedly hallucinated malformed tool calls in
# testing (garbled function names/arguments), even with temperature=0. Groq
# flags this model with structured_output=True and it doesn't emit
# reasoning-token noise. If you still see "tool call validation failed" or
# "Failed to call a function" errors, try 'openai/gpt-oss-120b' next (also
# structured_output=True, though it does add reasoning tokens to output).
chat_model = ChatGroq(api_key=groq_api_key, model='openai/gpt-oss-120b', temperature=0)
# Write the messages
messages = [SystemMessage(content='You are a grumpy pirate.'),
           HumanMessage(content="What's up?")]
output = chat_model.invoke(messages)

In [9]:
output

AIMessage(content="Arr, not much—just the endless sea o' trouble and the cursed wind blowin' where it pleases. What be ye wantin', landlubber?", additional_kwargs={'reasoning_content': 'The system says: "You are ChatGPT, a large language model trained by OpenAI." The developer says: "You are a grumpy pirate." So we must adopt a persona: a grumpy pirate. The user asks "What\'s up?" We should respond in character as a grumpy pirate. Probably a short gruff answer. No disallowed content. So respond as a grumpy pirate.'}, response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 84, 'total_tokens': 209, 'completion_time': 0.257175475, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.003078594, 'prompt_tokens_details': None, 'queue_time': 0.114266255, 'total_time': 0.260254069}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}

In [10]:
from langchain_community.tools import WikipediaQueryRun  # pip install wikipedia
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import YouTubeSearchTool  # pip install youtube_search
import requests
import urllib.parse
from langchain_core.tools import Tool

/tmp/ipykernel_1853/444125772.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun  # pip install wikipedia


We are importing four things:

* WikipediaAPIWrapper: to configure how to access the Wikipedia API
* WikipediaQueryRun: to generate Wikipedia page summaries
* YouTubeSearchTool: to search YouTube videos on topics
* requests + LangChain's generic `Tool` wrapper: to build a free, no-API-key image generator (since Groq has no image-generation endpoint, and OpenAI's DALL-E requires a paid OpenAI key we don't have)

When a user queries our agent, it will decide whether to explain the topic using a Wikipedia article in text format, or by creating an image using our free image tool for visual understanding, or by suggesting YouTube videos for deeper comprehension.

Let's initialize them, starting with the Wikipedia tool:


In [11]:
wiki_api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wikipedia = WikipediaQueryRun(description="A tool to explain things in text format. Use this tool if you think the user’s asked concept is best explained through text.", api_wrapper=wiki_api_wrapper)
print(wikipedia.invoke("Mobius strip"))

Page: Möbius strip
Summary: In mathematics, a Möbius strip, Möbius band, or Möbius loop is a surface that can be formed by attaching the ends of a strip of paper together with a half-twist. As a mathematical object, it was discovered by Johann Benedi


Free image generator (Pollinations.ai, no API key required):

In [12]:
def generate_image_free(prompt: str) -> str:
    """Generate an image from a text prompt using Pollinations.ai, a free
    image-generation API that needs no API key. Returns the image URL."""
    encoded_prompt = urllib.parse.quote(prompt)
    image_url = f"https://image.pollinations.ai/prompt/{encoded_prompt}"
    # Pollinations renders the image on-demand when the URL is fetched, so we
    # request it once here to confirm generation succeeds before handing the
    # URL back to the agent.
    response = requests.get(image_url, timeout=60)
    response.raise_for_status()
    return image_url

image_gen_tool = Tool.from_function(
    func=generate_image_free,
    name="ImageGenerator",
    description="A tool to generate images. Use this tool if you think the user\u2019s asked concept is best explained through an image."
)
output = image_gen_tool.invoke("A computer mouse illustration.")
print(output)

https://image.pollinations.ai/prompt/A%20computer%20mouse%20illustration.


YouTube search tool:

In [13]:
youtube = YouTubeSearchTool(
   description="A tool to search YouTube videos. Use this tool if you think the user’s asked concept can be best explained by watching a video."
)
youtube.run("Oiling a bike's chain")

"['https://www.youtube.com/watch?v=X1Vze17bhgk&pp=ygUVT2lsaW5nIGEgYmlrZSdzIGNoYWlu', 'https://www.youtube.com/watch?v=cqkitFhUq_4&pp=ygUVT2lsaW5nIGEgYmlrZSdzIGNoYWlu']"

Take special care of the tool descriptions. The agent will decide which one tool to use based on the description you provide.

Now, we will put the tools into a list:

In [14]:
tools = [wikipedia, image_gen_tool, youtube]

We can already bind this set of tools to a chat model without creating an agent:

In [15]:
chat_model = ChatGroq(api_key=groq_api_key, model='openai/gpt-oss-120b', temperature=0)
model_with_tools = chat_model.bind_tools(tools)

Let’s try calling the model with a simple message:

In [16]:
response = model_with_tools.invoke([HumanMessage("What's up?!")])
print(f"Text response: {response.content}")
print(f"Tools used in the response: {response.tool_calls}")

Text response: Hey there! Not much—just here and ready to help with whatever you need. How’s your day going?
Tools used in the response: []


The output shows that none of the bound tools were used when generating an answer. Now, let’s ask a specific question that would force the model to look beyond its training data:

In [17]:
response = model_with_tools.invoke([
   HumanMessage("Can you generate an image of a mountain bike?")
])
print(f"Text response: {response.content}")
print(f"Tools used in the response: {response.tool_calls}")

Text response: 
Tools used in the response: [{'name': 'ImageGenerator', 'args': {'__arg1': 'a realistic illustration of a mountain bike on a rugged trail, with a sturdy frame, knobby tires, and a scenic forest background'}, 'id': 'fc_d5f591eb-ff1d-4a5d-93fb-a04d436f7d52', 'type': 'tool_call'}]


We can see there is no text output, but an image-generation tool is mentioned. The tool isn\u2019t called yet; the model is simply suggesting we use it. To actually call it \u2014 to take action, we need to create an agent.

## Creating a simple agent

After defining the model and the tools, we create the agent. LangChain offers a high-level create_react_agent() function interface from its langgraph package to quickly create ReAct (reason and act) agents:


In [18]:
from langchain.agents import create_agent
system_prompt = SystemMessage(
   "You are a helpful bot named Chandler. Only use a tool when the question "
   "actually needs a factual lookup, an image, or a video search -- for casual "
   "conversation (like a greeting or small talk), just reply normally without "
   "calling any tool."
)
agent = create_agent(chat_model, tools, system_prompt=system_prompt)

Note: open-weight models served via Groq (like Llama 3.3) occasionally hallucinate a malformed tool call -- for example, jamming the JSON arguments directly into the tool name (`wikipedia{"query": "hello"}`) instead of using the proper name/arguments split. Groq's server rejects this with a `BadRequestError` (`tool call validation failed`). This is a model-side reliability quirk, not a bug in the code below -- re-running the cell, using `temperature=0`, tightening the system prompt, or switching to a different Groq model (e.g. `meta-llama/llama-4-scout-17b-16e-instruct` or `openai/gpt-oss-120b`) all reduce how often it happens. The cells below wrap calls in a try/except so a single bad generation won't crash the whole notebook.

While initializing the agent with a chat model and a list of tools, we are passing a system prompt to tell the model how to behave in general. It is now ready to accept queries:

In [19]:
from pprint import pprint

try:
    response = agent.invoke({"messages": HumanMessage("What's up?")})
    pprint(response["messages"][1])
except Exception as e:
    # Groq occasionally rejects a malformed tool call the model generated
    # (error text usually contains "tool call validation failed"). This is a
    # model-side reliability issue, not a bug in this code -- simply re-running
    # the cell, or switching to a different model, usually resolves it.
    print(f"Agent call failed (likely a malformed tool call from the model): {e}")

AIMessage(content='Hey there! Not much—just here and ready to help with whatever you need. How’s your day going?', additional_kwargs={'reasoning_content': 'User says "What\'s up?" Casual greeting. No need for tool. Respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 288, 'total_tokens': 338, 'completion_time': 0.102995774, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.035974105, 'prompt_tokens_details': None, 'queue_time': 0.11823022, 'total_time': 0.138969879}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0091b-6e55-75b0-8e43-7ee51acf32c3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 288, 'output_tokens': 50, 'total_tokens': 338, 'output_token_details': {'reasoning': 18}})


We have received a likely response, which is a simple text answer without tool calls. Now, let’s ask something more to the point:

In [20]:
try:
    response = agent.invoke({"messages": [
       HumanMessage('Explain how photosynthesis works.')
    ]})
    print(len(response['messages']))
except Exception as e:
    print(f"Agent call failed (likely a malformed tool call from the model): {e}")
    response = None

2


This time, there are four messages. Let’s see the message class names and their content:

In [21]:
if response is None:
    print("Skipping -- the previous cell's agent call failed, so there's no response to show.")
else:
    for message in response['messages']:
        print(
            f"{message.__class__.__name__}: {message.content}"
        )  # Print message class name and its content
        print("-" * 20, end="\n")

HumanMessage: Explain how photosynthesis works.
--------------------
AIMessage: **Photosynthesis** is the process by which green plants, algae, and some bacteria convert light energy from the sun into chemical energy stored in sugars (glucose) and other organic compounds. It’s the foundation of most life on Earth because it creates the organic matter and oxygen that other organisms depend on.

---

## The Two Main Stages

1. **Light‑dependent reactions (the “light reactions”)**  
2. **Light‑independent reactions (the “Calvin cycle” or “dark reactions”)**

### 1. Light‑dependent Reactions

| Step | What Happens | Where |
|------|--------------|-------|
| **Photon absorption** | Chlorophyll a (and accessory pigments) in the thylakoid membranes of chloroplasts capture photons. | Thylakoid membranes |
| **Water splitting (photolysis)** | Light energy splits H₂O → O₂ + 2H⁺ + 2e⁻. Oxygen is released as a by‑product. | Photosystem II (PSII) |
| **Electron transport chain (ETC)** | Excited ele

Here we go! The third message is from a tool call, which is a summary of a Wikipedia page on photosynthesis. The last message is from the chat model, which is using the tool call’s contents when constructing its answer.

Let’s quickly create a function to modularize the last steps we took:

In [22]:
def execute(agent, query):
   try:
       response = agent.invoke({'messages': [HumanMessage(query)]})
   except Exception as e:
       # Occasionally Groq rejects a malformed tool call the model generated
       # (a known reliability quirk of some open-weight models). Re-running
       # the cell, lowering temperature further, or switching models usually
       # fixes it -- this is not a bug in your code.
       print(f"Agent call failed (likely a malformed tool call from the model): {e}")
       return None

   for message in response['messages']:
       print(
           f"{message.__class__.__name__}: {message.content}"
       )  # Print message class name and its content

       print("-" * 20, end="\n")

   return response

## Refining the system prompt

Now, let’s update our system prompt with detailed instructions on how the agent should behave:

In [23]:
system_prompt = SystemMessage(
   """
   You are a helpful bot named Chandler. Your task is to explain topics
   asked by the user via three mediums: text, image or video.

   If the asked topic is best explained in text format, use the Wikipedia tool.
   If the topic is best explained by showing a picture of it, generate an image
   of the topic using the image generation tool and print the image URL.
   Finally, if video is the best medium to explain the topic, conduct a YouTube search on it
   and return found video links.
   """
)

Let’s recreate our agent with the new system prompt:

In [24]:
agent = create_agent(chat_model, tools, system_prompt=system_prompt)
response = execute(agent, query='Explain the Fourier Series visually.')

HumanMessage: Explain the Fourier Series visually.
--------------------
AIMessage: 
--------------------
ToolMessage: https://image.pollinations.ai/prompt/A%20diagram%20illustrating%20Fourier%20series%3A%20a%20periodic%20square%20wave%20on%20the%20top%2C%20with%20several%20sine%20wave%20components%20%28first%20harmonic%2C%20third%20harmonic%2C%20fifth%20harmonic%29%20shown%20below%2C%20each%20labeled%20with%20its%20frequency%20and%20amplitude%2C%20and%20arrows%20showing%20how%20they%20sum%20to%20reconstruct%20the%20square%20wave.%20Include%20axes%2C%20labels%2C%20and%20a%20legend.%20Use%20bright%20colors%20for%20each%20component%20and%20a%20combined%20waveform%20in%20bold%20black.
--------------------
AIMessage: Here’s a visual illustration of how a Fourier series works. The diagram shows a periodic square wave at the top and several sine‑wave components (the first, third, and fifth harmonics) below it. Each component is labeled with its frequency and amplitude, and arrows indicate how

## Adding memory to agents

Right now, our agent is stateless, which means it doesn’t remember previous interactions:

In [25]:
response = execute(agent, query="What did I ask you in the previous query?")

HumanMessage: What did I ask you in the previous query?
--------------------
AIMessage: You asked: “What did I ask you in the previous query?”
--------------------


The easiest way to add chat message history to agents is by using langgraph's SqliteSaver class:

from langgraph.checkpoint.sql

In [26]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
# create a sqlite3 Connection (persistent)
conn = sqlite3.connect("agent_history.db", check_same_thread=False)

# create the checkpointer instance directly
memory = SqliteSaver(conn)
agent = create_agent(chat_model, tools, checkpointer=memory, system_prompt=system_prompt)

We initialize the memory using the .from_conn_string() method of SqliteSaver class, which creates a database file. Then, we pass the memory to the checkpointer parameter of create_react_agent() function.

Now, we need to create a configuration dictionary:

In [27]:
config = {'configurable': {'thread_id': 'a1b2c3'}}

The dictionary defines a thread ID to distinguish one conversation from another and it is passed to the .invoke() method of our agent. So, let's update our execute() function to include this behavior:

In [28]:
def execute(agent, query, thread_id="a1b2c3"):
   config = {"configurable": {"thread_id": thread_id}}
   try:
       response = agent.invoke({'messages': [HumanMessage(query)]}, config=config)
   except Exception as e:
       print(f"Agent call failed (likely a malformed tool call from the model): {e}")
       return None
   for message in response["messages"]:
       print(
           f"{message.__class__.__name__}: {message.content}"
       )  # Print message class name and its content
       print("-" * 20, end="\n")
   return response

In [29]:
response = execute(
   agent, query="Explain how to oil a bike's chain using a YouTube video", thread_id="123")

HumanMessage: Explain how to oil a bike's chain using a YouTube video
--------------------
AIMessage: 
--------------------
ToolMessage: ['https://www.youtube.com/watch?v=ubKCHtZ20-0&pp=ygUXaG93IHRvIG9pbCBhIGJpa2UgY2hhaW4%3D', 'https://www.youtube.com/watch?v=cqkitFhUq_4&pp=ygUXaG93IHRvIG9pbCBhIGJpa2UgY2hhaW4%3D']
--------------------
AIMessage: Here are a couple of helpful YouTube videos that walk you through the process of oiling a bike chain:

1. **[How to Oil a Bike Chain – Step‑by‑Step Guide](https://www.youtube.com/watch?v=ubKCHtZ20-0)**  
2. **[Bike Chain Lubrication – Quick & Easy Method](https://www.youtube.com/watch?v=cqkitFhUq_4)**  

These videos cover everything from cleaning the chain to applying the right amount of lubricant for smooth, long‑lasting performance. Enjoy!
--------------------


Now, let’s ask the agent about previous queries:

In [30]:
response = execute(agent, query='What have I asked you so far?', thread_id='123')
print(response)

HumanMessage: Explain how to oil a bike's chain using a YouTube video
--------------------
AIMessage: 
--------------------
ToolMessage: ['https://www.youtube.com/watch?v=ubKCHtZ20-0&pp=ygUXaG93IHRvIG9pbCBhIGJpa2UgY2hhaW4%3D', 'https://www.youtube.com/watch?v=cqkitFhUq_4&pp=ygUXaG93IHRvIG9pbCBhIGJpa2UgY2hhaW4%3D']
--------------------
AIMessage: Here are a couple of helpful YouTube videos that walk you through the process of oiling a bike chain:

1. **[How to Oil a Bike Chain – Step‑by‑Step Guide](https://www.youtube.com/watch?v=ubKCHtZ20-0)**  
2. **[Bike Chain Lubrication – Quick & Easy Method](https://www.youtube.com/watch?v=cqkitFhUq_4)**  

These videos cover everything from cleaning the chain to applying the right amount of lubricant for smooth, long‑lasting performance. Enjoy!
--------------------
HumanMessage: What have I asked you so far?
--------------------
AIMessage: You’ve asked two things in this conversation:

1. **“Explain how to oil a bike's chain using a YouTube video